# Assignment 4 — Mini Myanmar ASR

| Part | Content |
|------|---------|
| **A** | Data preparation (speaker-held-out split) |
| **B** | Kaldi baseline — Monophone + Triphone HMM-GMM |
| **C** | Advanced models — DNN-HMM + BiLSTM-CTC + HF Wav2Vec2 fine-tune |
| **D** | Results table |
| **E** | **Testing demo** — see also `Assignment4_Demo.ipynb` + `demo/playlist.json` |

> **Metrics:** Myanmar has no reliable space-separated words here, so the lexicon uses **character tokens**.  
> **Char WER** = Word Error Rate under that tokenization. **SER** = Syllable Error Rate.


---
Data Preparation

**Goal:** Turn `raw-corpus/` recordings into Kaldi `train` / `dev` / `test`.

### A.1 Setup


In [ ]:
from pathlib import Path
import json
import subprocess
from IPython.display import display, Audio, Markdown, HTML

def find_root() -> Path:
    here = Path.cwd().resolve()
    candidates = [here, here.parent]
    if here.name == "notebooks":
        candidates.insert(0, here.parent)
    candidates.append(Path("/home/administrator/assignmen-kzyn/KyawZayaNaing-assignment4"))
    for cand in candidates:
        if (cand / "scripts" / "prepare_kaldi_data.py").exists():
            return cand.resolve()
    raise FileNotFoundError("Cannot find project root")

ROOT = find_root()
RAW = ROOT / "raw-corpus"
OUT = ROOT / "kaldi_data"
EXP = ROOT / "kaldi_recipe" / "exp"
DATA = ROOT / "kaldi_recipe" / "data"
LOGS = ROOT / "logs"
LOGS.mkdir(parents=True, exist_ok=True)

CONDA = Path.home() / "miniconda3" / "etc" / "profile.d" / "conda.sh"
ENV = "kaldi-asr"
SKIP_IF_DONE = False  # True = reuse finished stages; set False to force retrain

SCRIPTS = {
    "prep": ROOT / "scripts" / "prepare_kaldi_data.py",
    "gmm":  ROOT / "scripts" / "kaldi_mini_train.py",
    "dnn":  ROOT / "scripts" / "kaldi_dnn_train.py",
    "ctc":  ROOT / "scripts" / "pytorch_ctc_train.py",
    "hf":   ROOT / "scripts" / "hf_finetune_asr.py",
}

print("ROOT :", ROOT)
print("RAW  :", RAW, "| exists =", RAW.exists())
print("SKIP_IF_DONE =", SKIP_IF_DONE)
for k, p in SCRIPTS.items():
    print(f"  {k:4s} → {p.name} ({'OK' if p.exists() else 'MISSING'})")


In [ ]:
# Must match scripts/prepare_kaldi_data.py
SPLIT = {
    "train": ["HanZawHtet", "NawSayRiHtoo", "KhantHtay", "KyawZayaNaing"],
    "dev":   ["YinMon"],
    "test":  ["YeLinTun", "PunKhattgar"],  
}
GENDER = {
    "HanZawHtet": "male",
    "NawSayRiHtoo": "female",
    "KhantHtay": "male",
    "YeLinTun": "male",
    "YinMon": "female",
    "KyawZayaNaing": "male",
    "PunKhattgar": "female",
}

print("Notebook SPLIT:")
print(json.dumps(SPLIT, ensure_ascii=False, indent=2))

cfg_path = OUT / "split_config.json"
if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
    if cfg.get("SPLIT") == SPLIT:
        print("\nsplit_config.json matches. counts =", cfg.get("counts"))
    else:
        print("\nsplit_config.json is STALE — A.4 will regenerate it.")
        print(json.dumps(cfg.get("SPLIT", {}), ensure_ascii=False, indent=2))
else:
    print("\nNo split_config.json yet — A.4 will create it.")

print("\nTest genders:", {s: GENDER[s] for s in SPLIT["test"]})
assert set(GENDER[s] for s in SPLIT["test"]) >= {"male", "female"}
assert not (set(SPLIT["train"]) & set(SPLIT["test"]))
print("✓ Test is 1M+1F, no train/test speaker overlap")


### A.3 Helpers


In [ ]:
def run_py(script: Path, *args: str, log: str = "run.log", env: str | None = ENV) -> None:
    log_path = LOGS / log
    arg_s = " ".join(args)
    # CTC / HF need torch (+transformers) on miniconda base. Kaldi stages use kaldi-asr env.
    py = "python" if env else str(Path.home() / "miniconda3" / "bin" / "python")
    cmd = "\n".join([
        f'source "{CONDA}"',
        f"conda activate {env}" if env else "true",
        f'cd "{ROOT}"',
        f'export PATH="$HOME/miniconda3/envs/kaldi-asr/bin:$PATH"',
        f'{py} "{script}" {arg_s} 2>&1 | tee "{log_path}"',
    ])
    print("▶", script.name, arg_s)
    print("  log →", log_path)
    proc = subprocess.run(["bash", "-lc", cmd], text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"failed rc={proc.returncode}; see {log_path}")
    print("✓ OK\n")


def show_scores(model: str, split: str = "test") -> dict | None:
    p = EXP / model / f"decode_{split}" / "scores.json"
    if not p.exists():
        print(f"  {model}/{split}: (no scores yet)")
        return None
    s = json.loads(p.read_text(encoding="utf-8"))
    print(f"  {model}/{split}:  Char WER = {s['char_wer']*100:5.1f}%   SER = {s['syllable_er']*100:5.1f}%   n = {s.get('n_utt')}")
    return s


def done(*paths: Path) -> bool:
    return SKIP_IF_DONE and all(p.exists() for p in paths)


def assert_wavs_local() -> None:
    bad = []
    for split in ("train", "dev", "test"):
        for ln in (OUT / split / "wav.scp").read_text().splitlines():
            if not ln.strip():
                continue
            path = Path(ln.split(maxsplit=1)[1]).resolve()
            try:
                path.relative_to(ROOT)
            except ValueError:
                bad.append(str(path))
    if bad:
        raise SystemExit(f"External audio not allowed. Examples: {bad[:3]}")
    print("✓ All wav paths are inside this project")


### A.4 Build Kaldi data


In [ ]:
need_prep = True
cfg_path = OUT / "split_config.json"
if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
    if cfg.get("SPLIT") == SPLIT and (OUT / "train" / "wav.scp").exists():
        need_prep = False

if need_prep:
    run_py(SCRIPTS["prep"], log="01_prepare.log", env=None)
else:
    print("Skip prepare — already matches notebook SPLIT.")

print("=" * 60)
print((OUT / "split_plan.md").read_text(encoding="utf-8"))
cfg = json.loads((OUT / "split_config.json").read_text(encoding="utf-8"))
print("counts:", cfg["counts"])
assert_wavs_local()


### A.5 Integrity check


In [ ]:
def check_split(name: str) -> dict:
    d = OUT / name
    wav_ids = [ln.split()[0] for ln in (d / "wav.scp").read_text().splitlines() if ln.strip()]
    text_ids = [ln.split()[0] for ln in (d / "text").read_text().splitlines() if ln.strip()]
    u2s = dict(ln.split() for ln in (d / "utt2spk").read_text().splitlines() if ln.strip())
    assert set(wav_ids) == set(text_ids) == set(u2s), f"{name}: ID mismatch"
    assert len(wav_ids) == len(set(wav_ids)), f"{name}: duplicate IDs"
    speakers = sorted(set(u2s.values()))
    print(f"  {name:5s}  utts={len(wav_ids):4d}  speakers={speakers}")
    return {"n": len(wav_ids), "speakers": speakers}

print("Data integrity:")
stats = {s: check_split(s) for s in ("train", "dev", "test")}
leak = set(stats["train"]["speakers"]) & set(stats["test"]["speakers"])
assert not leak, f"TEST leaked into TRAIN: {leak}"
assert set(stats["test"]["speakers"]) == set(SPLIT["test"])
print("\n✓ Held-out OK — test speakers not in train")


---
# Part B — Kaldi HMM-GMM Baseline

| Stage | What it does |
|-------|----------------|
| 0–2 | Copy data, character lexicon, MFCC + CMVN |
| 3 | **Monophone** HMM-GMM |
| 4 | **Triphone** HMM-GMM |
| 5 | Decode test/dev → Char WER + SER |


### B.1 Features & lexicon


In [ ]:
feat_train = DATA / "train" / "feats.scp"
n_data = len((OUT / "train" / "utt2spk").read_text().splitlines())
n_feat = len(feat_train.read_text().splitlines()) if feat_train.exists() else -1

if done(feat_train) and n_data == n_feat:
    print(f"Skip features — already ready (train={n_feat})")
else:
    if n_feat >= 0 and n_data != n_feat:
        print(f"Feat count mismatch (data={n_data}, feats={n_feat}) — rebuilding")
    run_py(SCRIPTS["gmm"], "--stage", "0", "--stop-stage", "2", log="02_kaldi_feat.log")
print("feats ready:", feat_train.exists())


### B.2 Monophone HMM-GMM


In [ ]:
MONO_ITERS = 20
if done(EXP / "mono" / "final.mdl"):
    print("Skip monophone — final.mdl exists")
else:
    run_py(SCRIPTS["gmm"], "--stage", "3", "--stop-stage", "3",
           "--mono-iters", str(MONO_ITERS), log="03_kaldi_mono.log")
print("mono/final.mdl:", (EXP / "mono" / "final.mdl").exists())


### B.3 Triphone HMM-GMM


In [ ]:
TRI_ITERS = 15
if done(EXP / "tri1" / "final.mdl"):
    print("Skip triphone — final.mdl exists")
else:
    try:
        run_py(SCRIPTS["gmm"], "--stage", "4", "--stop-stage", "4",
               "--tri-iters", str(TRI_ITERS), log="04_kaldi_tri.log")
    except Exception as e:
        print("Triphone failed (optional):", e)
print("tri1/final.mdl:", (EXP / "tri1" / "final.mdl").exists())


### B.4 Decode & score (GMM)


In [ ]:
# Needs Setup (ROOT/SCRIPTS + run_py/show_scores)
if "run_py" not in globals() or "EXP" not in globals():
    raise RuntimeError("Run Setup cells first (paths + run_py), then re-run this cell.")

# Baseline for Part C is mono Char-WER. tri1 is optional.
mono_scores = EXP / "mono" / "decode_test" / "scores.json"
tri_scores = EXP / "tri1" / "decode_test" / "scores.json"
tri_ok = (EXP / "tri1" / "final.mdl").exists()
need_decode = not mono_scores.exists() or (tri_ok and not tri_scores.exists())

if SKIP_IF_DONE and not need_decode:
    print("Skip GMM decode — scores already present")
else:
    run_py(
        SCRIPTS["gmm"],
        "--stage", "5",
        "--stop-stage", "5",
        "--decode-split", "test",
        log="05_kaldi_decode.log",
    )

print("GMM baselines (test):")
show_scores("mono", "test")
show_scores("tri1", "test")
print("GMM (dev):")
show_scores("mono", "dev")
show_scores("tri1", "dev")


---
# Part C — Advanced Models

| Model | Idea |
|-------|------|
| **Kaldi DNN-HMM** | Preconditioned DNN on **mono** alignments (512×3) + char-bigram decode |
| **PyTorch CNN+CNN+BiLSTM-CTC** | CNN frontend + BiLSTM character CTC (from-scratch) |
| **HF Wav2Vec2 fine-tune** | Pretrained encoder + CTC head (Hugging Face) |

---

### C.1 Kaldi DNN-HMM


In [ ]:
# Needs Setup: cell with ROOT/SCRIPTS + cell with run_py/show_scores
if "run_py" not in globals() or "EXP" not in globals() or "SCRIPTS" not in globals():
    raise RuntimeError(
        "run_py / EXP not defined — run the Setup cells first "
        "(paths + SCRIPTS, then run_py/show_scores), then re-run this C.1 cell."
    )

mono_ok = (EXP / "mono" / "final.mdl").exists() and (EXP / "mono" / "final.ali").exists()
dnn_scores = EXP / "dnn" / "decode_test" / "scores.json"
dnn_mdl = EXP / "dnn" / "final.mdl"
mono_scores = EXP / "mono" / "decode_test" / "scores.json"
gmm_src = EXP / "dnn" / "gmm_source.txt"
used_mono = gmm_src.exists() and gmm_src.read_text(encoding="utf-8").strip() == "mono"

def _wer(path):
    if not path.exists():
        return None
    try:
        return float(json.loads(path.read_text(encoding="utf-8")).get("char_wer", 9))
    except Exception:
        return None

dnn_wer = _wer(dnn_scores)
mono_wer = _wer(mono_scores)
beats_mono = (dnn_wer is not None and mono_wer is not None and dnn_wer < mono_wer)

print(
    f"DNN mdl={dnn_mdl.exists()} gmm_src={'mono' if used_mono else (gmm_src.read_text().strip() if gmm_src.exists() else 'missing')} "
    f"scores_wer={dnn_wer} mono_wer={mono_wer} beats_mono={beats_mono}"
)

if not mono_ok:
    print("Skip DNN — need mono/final.mdl + final.ali first (Part B)")
elif dnn_scores.exists() and beats_mono and used_mono:
    print("DNN OK (mono alignments, beats GMM mono)")
    show_scores("dnn", "test")
elif dnn_mdl.exists() and used_mono:
    # Model already trained — decode only (do not wipe 35 epochs)
    print("→ Decode existing DNN (prior-adjust + char-bigram + as/lm tune)")
    if dnn_scores.exists():
        dnn_scores.unlink()
    run_py(
        SCRIPTS["dnn"],
        "--stage", "1",
        "--gmm-exp", "mono",
        "--decode-split", "test",
        "--use-gpu", "yes",
        log="06_kaldi_dnn_decode.log",
    )
    show_scores("dnn", "test")
else:
    print("→ Retrain DNN on mono alignments: 512×3 splice5 dropout0.1 + bigram decode")
    if dnn_scores.exists():
        dnn_scores.unlink()
    run_py(
        SCRIPTS["dnn"],
        "--stage", "0",
        "--gmm-exp", "mono",
        "--epochs", "40",
        "--hidden", "512",
        "--layers", "3",
        "--left-context", "5",
        "--right-context", "5",
        "--dropout", "0.1",
        "--minibatch", "256",
        "--use-gpu", "yes",
        log="06_kaldi_dnn.log",
    )
    show_scores("dnn", "test")


### C.2 PyTorch CNN+BiLSTM-CTC


In [ ]:
ctc_scores = EXP / "ctc" / "decode_test" / "scores.json"
ctc_ckpt = EXP / "ctc" / "best.pt"
if ctc_scores.exists():
    print("CTC scores present")
elif ctc_ckpt.exists():
    # Checkpoint exists but TEST scores missing → eval-only (no retrain)
    print("CTC best.pt exists, scores missing → eval-only")
    run_py(SCRIPTS["ctc"], "--eval-only", "--beam", "1", log="07_ctc_eval.log", env=None)
else:
    run_py(SCRIPTS["ctc"], log="07_ctc.log", env=None)
show_scores("ctc", "test")
print("checkpoint:", ctc_ckpt.exists())


### C.3 Hugging Face Wav2Vec2 fine-tuning

Pretrained `facebook/wav2vec2-base` → freeze feature encoder → CTC fine-tune on our character vocab (same train/dev/test).

In [ ]:
hf_scores = EXP / "hf_w2v2" / "decode_test" / "scores.json"
hf_best = EXP / "hf_w2v2" / "best" / "model.safetensors"
need_hf = True
if SKIP_IF_DONE and hf_scores.exists() and hf_best.exists():
    s = json.loads(hf_scores.read_text(encoding="utf-8"))
    if s.get("char_wer", 1.0) < 0.95:
        need_hf = False
        print("Skip HF fine-tune — checkpoint + usable scores exist")
if need_hf:
    run_py(
        SCRIPTS["hf"],
        "--epochs", "18",
        "--batch-size", "8",
        "--lr", "5e-5",
        "--freeze-transformer-layers", "8",
        log="08_hf_w2v2.log",
        env=None,
    )
show_scores("hf_w2v2", "test")
print("checkpoint:", (EXP / "hf_w2v2" / "best" / "model.safetensors").exists())


---
# Part D — Results
Final comparison on **held-out test** speakers: `YeLinTun` (M) + `PunKhattgar` (F).


### D.1 Score table


In [ ]:
rows = []
for name, key in [
    ("GMM Monophone", "mono"),
    ("GMM Triphone", "tri1"),
    ("Kaldi DNN-HMM", "dnn"),
    ("PyTorch CNN+BiLSTM-CTC", "ctc"),
    ("HF Wav2Vec2 FT", "hf_w2v2"),
]:
    p = EXP / key / "decode_test" / "scores.json"
    if not p.exists():
        rows.append((name, key, None, None, None))
        continue
    s = json.loads(p.read_text(encoding="utf-8"))
    rows.append((name, key, s.get("char_wer"), s.get("syllable_er"), s.get("n_utt")))

print(f"{'Model':<22} {'Char WER':>10} {'SER':>10} {'#utt':>6}")
print("-" * 52)
for name, key, wer, ser, n in rows:
    w = f"{wer*100:9.1f}%" if wer is not None else "      —"
    s = f"{ser*100:9.1f}%" if ser is not None else "      —"
    u = f"{n:6d}" if n is not None else "     —"
    print(f"{name:<22} {w} {s} {u}")

summary = [{"model": n, "char_wer": w, "syllable_er": s, "n_utt": u}
           for n, _, w, s, u in rows]
(EXP / "summary_test_scores.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
)

avail = [(n, k, w, s) for n, k, w, s, _ in rows if w is not None]
BEST_KEY = None
if avail:
    best = min(avail, key=lambda t: t[2])
    BEST_KEY = best[1]
    print(f"\n★ Best model: {best[0]}  ({best[2]*100:.1f}% Char WER, {best[3]*100:.1f}% SER)")
print("Wrote", EXP / "summary_test_scores.json")
